# Loading in HPC Data to Make Surrogate Model Surface Plots

## (1): Import Libraries:

In [ ]:
import datetime
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## (2): Matplotlib rcparams Configuration:

They're my favorite settings. That's all.

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Loading HPC Data:

### (3.1): Load "Discrete" Predictions of the DNN

The DNN has predictions for all its training data; this is what we mean by "discrete" predictions.

In [103]:
dnn_discrete_prediction_files = glob.glob("./data/replica_*_test_predictions.csv")
dnn_discrete_prediction_dataframes = [pd.read_csv(f) for f in dnn_discrete_prediction_files]

print(f"[INFO]: Read {len(dnn_discrete_prediction_dataframes)} discrete DNN predictions.")

[INFO]: Read 23 discrete DNN predictions.


### (3.2): Load "Smooth" Predictions of the DNN

We now ask the DNN to evaluate across a fine continuum of datapoints; that's called the "smooth" predictions.

In [104]:
smooth_dnn_prediction_files = glob.glob("./data/replica_*_smooth_predictions.csv")
smooth_dnn_prediction_dataframes = [pd.read_csv(f) for f in smooth_dnn_prediction_files]

print(f"[INFO]: Read {len(smooth_dnn_prediction_dataframes)} smooth DNN predictions.")

[INFO]: Read 23 smooth DNN predictions.


### (3.3): Must pass this assert:

In [ ]:
assert len(smooth_dnn_prediction_dataframes) == len(dnn_discrete_prediction_dataframes), "[ASSERT]: Two prediction files do not have matching lengths."

## (4): Replica Data Loading:

### (4.1): Number of Replicas:

In [ ]:
# compute the number of replicas:
number_of_replicas = len(dnn_discrete_prediction_dataframes)

print(f"[INFO]: Total replicas in this batch: {number_of_replicas}")

### (4.2): Discrete Replica Predictions:

In [ ]:
discrete_predictions = np.stack([ df[['pred_xsec', 'pred_bsa']].values for df in dnn_discrete_prediction_dataframes ])

average_discrete_prediction = discrete_predictions.mean(axis = 0)
standard_dev_discrete_prediction  = discrete_predictions.std(axis = 0)

### (4.3): Smooth Replica Predictions:

In [ ]:
smooth_predictions = np.stack([ df[['pred_xsec', 'pred_bsa']].values for df in smooth_dnn_prediction_dataframes ])

average_smooth_prediction = smooth_predictions.mean(axis = 0)
standard_dev_smooth_prediction  = smooth_predictions.std(axis = 0)

### (4.4): Discrete Replica Dataframe Statistics:

In [ ]:
discrete_results = dnn_discrete_prediction_dataframes[0].copy()

discrete_results['xsec_mean'] = average_discrete_prediction[:, 0]
discrete_results['xsec_std'] = standard_dev_discrete_prediction[:, 0]

discrete_results['bsa_mean'] = average_discrete_prediction[:, 1]
discrete_results['bsa_std'] = standard_dev_discrete_prediction[:, 1]

### (4.5): Smooth Replica Dataframe Statistics:

In [ ]:
smooth_results = smooth_dnn_prediction_dataframes[0].copy()

smooth_results['xsec_mean'] = average_smooth_prediction[:, 0]
smooth_results['xsec_std'] = standard_dev_smooth_prediction[:, 0]

smooth_results['bsa_mean'] = average_smooth_prediction[:, 1]
smooth_results['bsa_std'] = standard_dev_smooth_prediction[:, 1]

### (4.6): Group by Kinematic Setting:

In [ ]:
discrete_grouped = discrete_results.groupby(['t', 'x_b', 'q_squared'], sort = True)
smooth_grouped = smooth_results.groupby(['t', 'x_b', 'q_squared'], sort = True)

## (5): Plotting Performance:

### (5.1): **The Main Plotting Loop**: Plots DNN smooth and discrete prediction for every kinematic setting:X

In [ ]:
# this is TRENTO CONVENTION: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361)

for kinematics, group in discrete_grouped:

    t_value, xb_value, qsquared_value = kinematics

    print(f"[INFO]: Current kinematic set: t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values('phi')

    # need these errors from the data...
    xsec_err = group['xsec_err'].values
    bsa_err = group['bsa_err'].values

    phi_vals = group['phi'].values

    true_xsec = group['true_xsec'].values
    true_bsa = group['true_bsa'].values

    xsec_mean = group['xsec_mean'].values
    xsec_std = group['xsec_std'].values

    bsa_mean = group['bsa_mean'].values
    bsa_std = group['bsa_std'].values

    smooth_group = smooth_grouped.get_group(kinematics)
    smooth_group = smooth_group.sort_values('phi')

    phi_smooth = smooth_group['phi'].values

    # smooth BSA cross-section interpolation
    xsec_smooth_mean = smooth_group['xsec_mean'].values
    xsec_smooth_std = smooth_group['xsec_std'].values

    # smooth BSA DNN interpolation
    bsa_smooth_mean = smooth_group['bsa_mean'].values
    bsa_smooth_std = smooth_group['bsa_std'].values
    
    phi = group['phi'].values

    # compute the residuals, the "e"s
    residuals_xsec = true_xsec - xsec_mean
    residuals_bsa = true_bsa - bsa_mean

    # this is chi_{nu}^{2}, i.e. reduced chi squared --- read about it on Wikipedia:
    reduced_chi_squared_xsec = np.sum(residuals_xsec**2) / len(phi)
    reduced_chi_squared_bsa = np.sum(residuals_bsa**2) / len(phi)

    # now we actually do plotting!
    residuals_figure, residuals_axes = plt.subplots(
        nrows = 2, ncols = 2, figsize = (10, 8), sharex = 'col', layout = "tight")

    # this plot is the DNN smooth interpolation for CROSS-SECTION:
    residuals_axes[0, 0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    residuals_axes[0, 0].fill_between(
        phi_smooth, xsec_smooth_mean - xsec_smooth_std, xsec_smooth_mean + xsec_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    residuals_axes[0, 0].errorbar(
        phi, true_xsec, yerr = xsec_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    residuals_axes[0, 0].set_ylabel(r"$d^{4}\sigma$", fontsize = 14)
    residuals_axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {reduced_chi_squared_xsec:.3f}$)")
    residuals_axes[0, 0].legend()
    residuals_axes[0, 0].grid(True, linestyle = ':', alpha = 0.6)

    # this plot is the DISCRETE RESIDUALS for the CROSS-SECTION:
    residuals_axes[0, 1].scatter(phi, residuals_xsec, color = 'blue', alpha = 0.6)
    residuals_axes[0, 1].axhline(0, color = 'black', linestyle = '--')
    residuals_axes[0, 1].set_title("Residuals")
    residuals_axes[0, 1].grid(True, linestyle = ':', alpha = 0.6)

    # this plot is the DNN smooth interpolation for the BEAM SPIN ASYMMETRY:
    residuals_axes[1, 0].plot(phi_smooth, bsa_smooth_mean, color = 'green', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    residuals_axes[1, 0].fill_between(
        phi_smooth, bsa_smooth_mean - bsa_smooth_std, bsa_smooth_mean + bsa_smooth_std,
        color = 'green', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    residuals_axes[1, 0].errorbar(
        phi, true_bsa, yerr = bsa_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    
    residuals_axes[1, 0].set_ylabel("BSA", fontsize = 14)
    residuals_axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    residuals_axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {reduced_chi_squared_bsa:.3f}$)")
    residuals_axes[1, 0].legend()
    residuals_axes[1, 0].grid(True, linestyle = ':', alpha = 0.6)

     # this plot is the DISCRETE RESIDUALS for the BEAM SPIN ASYMMETRY:
    residuals_axes[1, 1].scatter(phi, residuals_bsa, color = 'purple', alpha = 0.6)
    residuals_axes[1, 1].axhline(0, color = 'black', linestyle = '--')
    residuals_axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    residuals_axes[1, 1].set_title("Residuals")
    residuals_axes[1, 1].grid(True, linestyle = ':', alpha = 0.6)

    residuals_axes[1, 0].text(
        -0.1, -0.1, 
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = residuals_axes[1, 0].transAxes)
    
    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$t = {t_value:.3f}$, $x_\textrm{{B}} = {xb_value:.3f}$, $Q^2={qsquared_value:.3f}$",
        fontsize = 16
    )

    # dynamically compute the file name:
    filename = f"./plots/t{t_value:.3f}_xb{xb_value:.3f}_q2{qsquared_value:.3f}_residuals_v1"

    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(residuals_figure)

    del residuals_axes
    del residuals_figure

### (5.2): Preparing to Plot DNN Surface across $\phi$ and $t$:

In [ ]:
xb_q2_groups = discrete_results.groupby(['x_b', 'q_squared'])
number_of_xb_qsquared_groups = xb_q2_groups.ngroups
print(f"[INFO]: There are {number_of_xb_qsquared_groups} (xb, Q^2) values once we take out t.")

phi_grid = np.linspace(-np.pi, np.pi, 361)

### (5.3): **The Second Main Plotting Loop**: Plots DNN surface plots across $\phi$ and $t$:

In [ ]:
for (xb_value, qsquared_value), group in xb_q2_groups:

    print(f"[INFO]: Generating surface plots: xb = {xb_value}, Q^2 = {qsquared_value}")

    smooth_subset = smooth_results[(smooth_results['x_b'] == xb_value) & (smooth_results['q_squared'] == qsquared_value)]

    group = group.sort_values(['t', 'phi'])

    phi_data = group['phi'].values
    t_data = group['t'].values

    true_xsec = group['true_xsec'].values
    true_bsa = group['true_bsa'].values

    xsec_mean = group['xsec_mean'].values
    xsec_std = group['xsec_std'].values

    bsa_mean = group['bsa_mean'].values
    bsa_std = group['bsa_std'].values

    # residuals
    residuals_xsec = true_xsec - xsec_mean
    residuals_bsa = true_bsa  - bsa_mean

    colors_xsec = np.where(residuals_xsec >= 0, 'red', 'blue')
    colors_bsa = np.where(residuals_bsa  >= 0, 'red', 'blue')

    smooth_group = smooth_results[
        (smooth_results['x_b'] == xb_value) &
        (smooth_results['q_squared'] == qsquared_value)
    ]

    smooth_group = smooth_group.sort_values(['t', 'phi'])

    # mesh coordinates
    t_values = np.sort(smooth_group['t'].unique())
    phi_values = np.sort(smooth_group['phi'].unique())
    phi_meshgrid, t_meshgrid = np.meshgrid(phi_values, t_values)

    n_t = len(t_values)
    n_phi = len(phi_values)

    # sigma vs. t and phi
    xsec_surface = smooth_group['xsec_mean'].values.reshape(n_t, n_phi)
    xsec_stddev_surface = smooth_group['xsec_std'].values.reshape(n_t, n_phi)

    # BSA vs. t and phi
    bsa_surface = smooth_group['bsa_mean'].values.reshape(n_t, n_phi)
    bsa_stddev_surface = smooth_group['bsa_std'].values.reshape(n_t, n_phi)

    # residual planes
    zero_plane_xsec = np.zeros_like(xsec_surface)
    zero_plane_bsa  = np.zeros_like(bsa_surface)

    surface_plots_figure = plt.figure(figsize = (10, 10), layout = "tight")

    ax1 = surface_plots_figure.add_subplot(2, 2, 1, projection = '3d')
    ax2 = surface_plots_figure.add_subplot(2, 2, 2, projection = '3d')
    ax3 = surface_plots_figure.add_subplot(2, 2, 3, projection = '3d')
    ax4 = surface_plots_figure.add_subplot(2, 2, 4, projection = '3d')

    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface, cmap = 'viridis', alpha = 0.5)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface + xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface - xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.scatter(phi_data, t_data, true_xsec, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax1.set_xlabel(r'$\phi$ (Radians)', fontsize = 16)
    ax1.set_ylabel(r'$t$', fontsize = 16)
    ax1.set_zlabel(r'$d^{4}\sigma^{UU}$', fontsize = 16)
    ax1.set_title(r'$d^{4}\sigma^{UU}$ Surface in $\phi$ and $t$', fontsize = 16)

    ax2.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_xsec, color = 'gray', alpha = 0.15)
    ax2.scatter(phi_data, t_data, residuals_xsec, color = colors_xsec, s = 20)
 
    ax2.set_xlabel(r'$\phi$', fontsize = 16)
    ax2.set_ylabel(r'$t$', fontsize = 16)
    ax2.set_zlabel('Residuals', fontsize = 16)
    ax2.set_title('Cross Section Residuals', fontsize = 16)

    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface, cmap='plasma', alpha = 0.5)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface + bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface - bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.scatter(phi_data, t_data, true_bsa, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax3.set_xlabel(r'$\phi$ (Radians)', fontsize = 16)
    ax3.set_ylabel(r'$t$', fontsize = 16)
    ax3.set_zlabel('BSA', fontsize = 16)
    ax3.set_title(r'BSA Surface in $\phi$ and $t$', fontsize = 16)

    ax4.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_bsa, color = 'gray', alpha = 0.15)
    ax4.scatter(phi_data, t_data, residuals_bsa, color = colors_bsa, s = 20)

    ax4.set_xlabel(r'$\phi$', fontsize = 16)
    ax4.set_ylabel(r'$t$', fontsize = 16)
    ax4.set_zlabel('Residuals', fontsize = 16)
    ax4.set_title('BSA Residuals', fontsize = 16)

    surface_plots_figure.suptitle(
        "Kinematic Setting\n"
        rf"$x_\textrm{{B}} = {xb_value:.4g}$, $Q^2 = {qsquared_value:.4g}$", fontsize = 18)

    plot_filename = f"./plots/surface_xb{xb_value:.4g}_q2{qsquared_value:.4g}"
    for extension in ['png', 'eps']:
        surface_plots_figure.savefig(f"{plot_filename}.{extension}", facecolor = 'white')

    plt.close(surface_plots_figure)

    del ax1
    del ax2
    del ax3
    del ax4
    del surface_plots_figure